# Figure demo

The cell below is tagged `#| label: figDemo`, which is what lets
`02-interactive-figures.md` embed its output with `:::{figure} #figDemo`.

This is Plotly's standard slider example. Replace it with your own work.

In [ ]:
#| label: figDemo

import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

# Emit BOTH representations of every figure:
#   application/vnd.plotly.v1+json  -> the interactive version, used on the website
#   image/png                        -> a static snapshot, used in the PDF
# A renderer picks whichever suits the medium. Without the "+png" half, a PDF export
# gets only a caption, because a slider is JavaScript and print cannot run it.
# The PNG shows whichever frame is visible by default, so make that the frame worth
# printing.
#
# Static export needs kaleido AND a Chrome install (`plotly_get_chrome`). Probe for it
# rather than assuming: if it is missing, fall back to interactive-only instead of
# raising, which would halt the whole notebook and wreck the build.
try:
    pio.to_image(go.Figure(), format="png")
    pio.renderers.default = "plotly_mimetype+png"
except Exception as exc:
    pio.renderers.default = "plotly_mimetype"
    print(f"Static PNG export unavailable ({type(exc).__name__}); "
          "the website is unaffected, but PDF figures will be captions only.")

x = np.arange(0, 10, 0.01)
freqs = np.arange(0, 5, 0.1)

fig = go.Figure()

# One trace per slider position. All hidden to start with.
for f in freqs:
    fig.add_trace(go.Scatter(
        visible=False,
        line=dict(color="#00CED1", width=4),
        name=f"freq = {f:.1f}",
        x=x,
        y=np.sin(f * x),
    ))

# Show one of them by default.
DEFAULT = 10
fig.data[DEFAULT].visible = True

# Each step makes exactly one trace visible. This is the whole mechanism:
# the slider does not compute anything, it chooses between precomputed frames.
steps = []
for i, f in enumerate(freqs):
    visible = [False] * len(fig.data)
    visible[i] = True
    steps.append(dict(method="update", args=[{"visible": visible}], label=f"{f:.1f}"))

fig.update_layout(
    sliders=[dict(active=DEFAULT, steps=steps, pad={"t": 50},
                  currentvalue={"prefix": "Frequency: "})],
    xaxis_title="x",
    yaxis_title="sin(freq · x)",
    yaxis_range=[-1.1, 1.1],
    width=760, height=460,
    margin=dict(l=60, r=30, t=30, b=80),
    template="plotly_white",
    showlegend=False,
)

fig
